## NIS2-NIST semantic matching

Pipeline note:
- this notebook is the canonical matching phase
- input technical requirements come from `nis2_only_technical_mpnet.csv`
- the final mapping method is embedding-based semantic similarity with E5


In [ ]:
!pip install -q sentence-transformers umap-learn


📌 1. Load NIS2 and NIST datasets


In [ ]:
import pandas as pd

df_nis2 = pd.read_csv("nis2_only_technical_mpnet.csv")
df_nist = pd.read_csv("nist_controls_svo_v2_with_family.csv")

print(f"✅ Loaded {len(df_nis2)} technical NIS2 requirements")
print(f"✅ Loaded {len(df_nist)} NIST controls")


📌 2. Generate E5 embeddings


In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "intfloat/e5-large"
model = SentenceTransformer(model_name)
print(f"✅ Loaded model: {model_name}")

nis2_texts = ["passage: " + text.strip() for text in df_nis2["Cleaned Requirement"]]
nist_texts = ["passage: " + text.strip() for text in df_nist["Cleaned Text"]]

nis2_embeddings = model.encode(
    nis2_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_tensor=True,
)
nist_embeddings = model.encode(
    nist_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_tensor=True,
)

print("✅ Embeddings generated for NIS2 and NIST datasets.")


📌 3. Save embeddings for reuse


In [ ]:
import numpy as np

np.save("nis2_embeddings_e5.npy", nis2_embeddings.cpu().numpy())
np.save("nist_embeddings_e5.npy", nist_embeddings.cpu().numpy())

print("✅ Embeddings saved to 'nis2_embeddings_e5.npy' and 'nist_embeddings_e5.npy'.")


📌 4. Perform semantic similarity matching


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

SIM_THRESHOLD = 0.83  # to be validated on a labelled sample

nis2_emb = np.load("nis2_embeddings_e5.npy")
nist_emb = np.load("nist_embeddings_e5.npy")

results = []

for i, req_row in df_nis2.iterrows():
    sims = cosine_similarity([nis2_emb[i]], nist_emb)[0]
    matched_idxs = np.where(sims >= SIM_THRESHOLD)[0]

    matched_controls = []
    matched_families = set()

    for idx in matched_idxs:
        control_id = df_nist.iloc[idx]["Control ID"]
        family = df_nist.iloc[idx]["Control Family"]
        score = float(sims[idx])
        matched_controls.append((control_id, round(score, 4)))
        matched_families.add(family)

    matched_controls.sort(key=lambda item: item[1], reverse=True)

    results.append({
        "Req ID": req_row["Req ID"],
        "Cleaned Requirement": req_row["Cleaned Requirement"],
        "Matched Controls": matched_controls,
        "Matched Families": sorted(matched_families),
    })

df_result = pd.DataFrame(results)
output_file = "nis2_nist_semantic_matches.csv"
df_result.to_csv(output_file, index=False)

print(f"✅ Saved semantic matching results to '{output_file}'.")
print(df_result.head())


📌 5. Inspect similarity score distribution


In [ ]:
import matplotlib.pyplot as plt

all_scores = []
for i in range(len(nis2_emb)):
    sims = cosine_similarity([nis2_emb[i]], nist_emb)[0]
    all_scores.extend(sims.tolist())

plt.figure(figsize=(8, 4.5))
plt.hist(all_scores, bins=50, color="steelblue", edgecolor="white")
plt.axvline(SIM_THRESHOLD, color="red", linestyle="--", label=f"Threshold = {SIM_THRESHOLD}")
plt.title("Distribution of cosine similarities (NIS2 vs NIST)")
plt.xlabel("Cosine similarity")
plt.ylabel("Number of pairs")
plt.legend()
plt.tight_layout()
plt.show()


📌 6. Export simplified mapping (Article -> Control IDs)


In [ ]:
import ast

df_simple = df_result[["Req ID", "Matched Controls"]].copy()

def extract_control_ids(value):
    matches = value if isinstance(value, list) else ast.literal_eval(value)
    return ", ".join(control_id for control_id, _ in matches)

df_simple["Control IDs"] = df_simple["Matched Controls"].apply(extract_control_ids)
df_simple[["Req ID", "Control IDs"]].to_csv("e5_nis2_to_nist_control_ids.csv", index=False)

print("✅ Saved simplified mapping to 'e5_nis2_to_nist_control_ids.csv'.")


📌 7. Visualize embedding space


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

images_dir = Path("images")
images_dir.mkdir(parents=True, exist_ok=True)

x_nis2 = np.load("nis2_embeddings_e5.npy")
x_nist = np.load("nist_embeddings_e5.npy")

x = np.vstack([x_nis2, x_nist])
y = np.array(["NIS2"] * len(x_nis2) + ["NIST"] * len(x_nist))

try:
    import umap

    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
    )
    emb2d = reducer.fit_transform(x)
    method = "UMAP"
except Exception:
    from sklearn.manifold import TSNE

    reducer = TSNE(
        n_components=2,
        perplexity=30,
        metric="cosine",
        random_state=42,
    )
    emb2d = reducer.fit_transform(x)
    method = "t-SNE"

fig, ax = plt.subplots(figsize=(7.2, 6))
for label in ["NIS2", "NIST"]:
    mask = y == label
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1], s=12, alpha=0.7, label=label)

ax.set_title(f"2D projection of embeddings ({method})")
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")
ax.grid(True, linestyle=":", linewidth=0.5)
ax.legend()

out = images_dir / "embeddings_2d_projection.png"
fig.savefig(out, dpi=220, bbox_inches="tight")
print("✅ Saved:", out)
plt.show()
